In [1]:
# 1. Installation
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers accelerate bitsandbytes pandas tqdm

# 2. Clonage du dépôt et déplacement
import os
!git clone https://github.com/abderrahmanamaoulouden533-ops/projet_rag_sujet6.git
os.chdir('/content/projet_rag_sujet6')

# 3. Téléchargement du corpus
import urllib.request
os.makedirs('corpus', exist_ok=True)
urls = {
    "doc_01_intro.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/getting_started/intro_tutorials/01_table_oriented.rst",
    "doc_02_read_write.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/getting_started/intro_tutorials/02_read_write.rst",
    "doc_03_selection.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/getting_started/intro_tutorials/03_subset_data.rst",
    "doc_04_plotting.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/getting_started/intro_tutorials/04_plotting.rst",
    "doc_05_10min.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/user_guide/10min.rst",
    "doc_06_io.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/user_guide/io.rst",
    "doc_07_indexing.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/user_guide/indexing.rst",
    "doc_08_merge.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/user_guide/merging.rst",
    "doc_09_groupby.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/user_guide/groupby.rst",
    "doc_10_reshape.rst": "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/source/user_guide/reshaping.rst"
}
for nom, url in urls.items():
    urllib.request.urlretrieve(url, f'corpus/{nom}')
print("✅ Environnement prêt et corpus téléchargé !")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Cloning into 'projet_rag_sujet6'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 12 (delta 3), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), done.
Resolving deltas: 100% (3/3), done.
✅ Environnement prêt et corpus téléchargé !


In [17]:
import pandas as pd

csv_text = """id|question|categorie|reponse_attendue|passages_sources|redacteur
Q01|Comment créer un DataFrame à partir d'un dictionnaire Python ?|present|pd.DataFrame({'A':[1,2],'B':[3,4]})|doc_01|MembreA
Q02|Quelle fonction permet de lire un fichier CSV avec un séparateur personnalisé ?|present|pd.read_csv('fichier.csv', sep=';')|doc_02|MembreB
Q03|Quelle méthode permet de sélectionner les lignes d'un DataFrame par étiquette d'index ?|present|.loc[]|doc_07|MembreC
Q04|Quelle méthode permet de sélectionner des lignes par position numérique ?|present|.iloc[]|doc_07|MembreA
Q05|Quelle fonction permet de fusionner deux DataFrames sur une clé commune ?|present|pd.merge(left, right, on='clé')|doc_08|MembreB
Q06|Comment calculer la moyenne d'une colonne numérique ?|present|df['col'].mean()|doc_05|MembreC
Q07|Comment grouper un DataFrame par une colonne et calculer une agrégation ?|present|df.groupby('col').agg(...)|doc_09|MembreA
Q08|Comment remplacer les valeurs manquantes d'un DataFrame ?|present|df.fillna(valeur) ou df.dropna()|doc_11|MembreB
Q09|Comment pivoter un DataFrame du format long vers le format large ?|present|df.pivot() ou df.pivot_table()|doc_10|MembreC
Q10|Comment créer un graphique à partir d'un DataFrame ?|present|df.plot()|doc_04|MembreA
Q11|Comment exporter un DataFrame vers un fichier Excel ?|present|df.to_excel('fichier.xlsx')|doc_02|MembreB
Q12|Comment trier un DataFrame selon une colonne ?|present|df.sort_values(by='col')|doc_05|MembreC
Q13|Comment concaténer deux DataFrames verticalement ?|present|pd.concat([df1, df2], axis=0)|doc_08|MembreA
Q14|Comment filtrer les lignes dont une colonne dépasse un seuil ?|present|df[df['col'] > seuil]|doc_07|MembreB
Q15|Quelle fonction permet de lire un fichier JSON dans pandas ?|present|pd.read_json('fichier.json')|doc_06|MembreC
Q16|Comment renommer les colonnes d'un DataFrame ?|present|df.rename(columns={'ancien':'nouveau'})|doc_05|MembreA
Q17|Comment convertir une colonne en type datetime ?|present|pd.to_datetime(df['col'])|doc_12|MembreB
Q18|Comment compter le nombre de valeurs uniques dans une colonne ?|present|df['col'].nunique()|doc_05|MembreC
Q19|Quelle option permet d'afficher plus de colonnes dans un DataFrame ?|present|pd.set_option('display.max_columns', N)|doc_14|MembreA
Q20|Comment supprimer une colonne d'un DataFrame ?|present|df.drop(columns=['col'])|doc_05|MembreB
Q21|Quelle est la politique de pandas concernant le support de MongoDB ?|absent|||MembreC
Q22|Quel est le salaire moyen d'un développeur pandas en France ?|absent|||MembreA
Q23|Quelle est la météo à Paris le jour de la sortie de pandas 2.0 ?|absent|||MembreB
Q24|Comment installer pandas sur un ordinateur quantique ?|absent|||MembreC
Q25|Quel est le nombre exact de contributeurs au projet pandas en 2020 ?|absent|||MembreA
Q26|Quelle est la durée légale du travail pour un data scientist ?|absent|||MembreB
Q27|Comment configurer pandas pour utiliser un GPU AMD ?|absent|||MembreC
Q28|Quelle est la capitale de l'Australie ?|absent|||MembreA
Q29|Quelle est la différence entre pandas et le framework Spark RDD ?|absent|||MembreB
Q30|Quel est l'impact carbone de l'exécution d'un script pandas ?|absent|||MembreC
Q31|Comment récupérer les données de la Bourse de Casablanca en temps réel ?|absent|||MembreA
Q32|Quel est le prix d'une licence commerciale pandas ?|absent|||MembreB
Q33|Comment intégrer pandas avec un modèle de deep learning dans TensorFlow ?|absent|||MembreC
Q34|Quelle est l'origine du nom "pandas" ?|absent|||MembreA
Q35|Comment fonctionne la gestion mémoire en C sous-jacente à pandas ?|absent|||MembreB
Q36|Comment faire ?|ambigu|||MembreC
Q37|Pourquoi ça ne marche pas ?|ambigu|||MembreA
Q38|Quelle est la meilleure façon de procéder ?|ambigu|||MembreB
Q39|Comment améliorer les performances ?|ambigu|||MembreC
Q40|Comment corriger cette erreur ?|ambigu|||MembreA
Q41|C'est quoi le problème avec mes données ?|ambigu|||MembreB
Q42|Comment utiliser cette fonction ?|ambigu|||MembreC
Q43|Comment faire pour que ce soit plus rapide ?|ambigu|||MembreA
Q44|Quelle syntaxe utiliser ?|ambigu|||MembreB
Q45|Comment gérer ce cas ?|ambigu|||MembreC
Q46|Comment lire un fichier CSV puis grouper les données par une colonne et calculer la moyenne d'une autre colonne ?|multi|pd.read_csv + groupby().mean()|doc_02 + doc_09|MembreA
Q47|Comparer les méthodes .loc et .iloc pour la sélection de lignes et de colonnes dans un DataFrame|multi|.loc = labels; .iloc = positions|doc_05 + doc_07|MembreB
Q48|Comment gérer les valeurs manquantes dans un DataFrame avant de générer un graphique ?|multi|fillna/dropna puis .plot()|doc_04 + doc_11|MembreC
Q49|Comment fusionner deux DataFrames puis pivoter le résultat pour produire un tableau croisé ?|multi|merge() puis pivot_table()|doc_08 + doc_10|MembreA
Q50|Comment convertir une chaîne en date puis extraire l'année et agréger par mois ?|multi|to_datetime puis .dt.year et resample|doc_09 + doc_12|MembreB
Q51|Comment exporter un DataFrame en CSV puis le recharger en préservant les types numériques et les dates ?|multi|to_csv + read_csv(parse_dates=..., dtype=...)|doc_02 + doc_06|MembreC
Q52|Comment détecter les doublons après une fusion de deux DataFrames et les supprimer ?|multi|merge + duplicated + drop_duplicates|doc_08 + doc_11|MembreA
Q53|Comment visualiser l'évolution temporelle d'une série groupée par catégorie ?|multi|groupby + .plot()|doc_09 + doc_13|MembreB
Q54|Comment concaténer deux DataFrames de structures différentes puis remplacer les NaN résultants ?|multi|pd.concat + fillna|doc_08 + doc_11|MembreC
Q55|Comment pivoter un DataFrame puis tracer un graphique en barres empilées ?|multi|pivot_table puis .plot(kind='bar')|doc_10 + doc_13|MembreA
Q56|Comment rééchantillonner une série temporelle et calculer la moyenne par mois ?|multi|resample('M').mean()|doc_09 + doc_12|MembreB
Q57|Comment filtrer un DataFrame sur plusieurs conditions et vérifier les valeurs manquantes du sous-ensemble ?|multi|df[(cond1) & (cond2)] puis .isna().sum()|doc_07 + doc_11|MembreC
Q58|Comment renommer les colonnes d'un DataFrame puis les utiliser comme index ?|multi|rename() puis set_index()|doc_05 + doc_07|MembreA
Q59|Comment lire un fichier JSON volumineux et le transformer en DataFrame pour une analyse groupée ?|multi|read_json + groupby|doc_06 + doc_09|MembreB
Q60|Comment désactiver les avertissements de pandas et afficher toutes les colonnes pour le débogage ?|multi|warnings.filterwarnings + pd.set_option|doc_05 + doc_14|MembreC"""

with open('questions.csv', 'w', encoding='utf-8') as f:
    f.write(csv_text)

# LA LIGNE CRUCIALE : sep='|' et keep_default_na=False
df_questions = pd.read_csv('questions.csv', sep='|', keep_default_na=False)
print(f"✅ {len(df_questions)} questions chargées avec succès.")
print(df_questions.head(2))

✅ 60 questions chargées avec succès.
    id                                           question categorie  \
0  Q01  Comment créer un DataFrame à partir d'un dicti...   present   
1  Q02  Quelle fonction permet de lire un fichier CSV ...   present   

                      reponse_attendue passages_sources redacteur  
0  pd.DataFrame({'A':[1,2],'B':[3,4]})           doc_01   MembreA  
1  pd.read_csv('fichier.csv', sep=';')           doc_02   MembreB  


In [3]:
import re, time, numpy as np, faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def charger_corpus(dossier="corpus"):
    docs = []
    for fname in sorted(os.listdir(dossier)):
        if fname.endswith(".rst"):
            with open(os.path.join(dossier, fname), encoding="utf-8") as f:
                docs.append({"source": fname, "texte": f.read()})
    return docs

def nettoyer_texte_rst(texte):
    texte = re.sub(r'\.\. (ipython|code-block|raw)::.*?(?=\n\n|\Z)', '', texte, flags=re.DOTALL)
    texte = re.sub(r'\.\. .*?::', '', texte)
    texte = re.sub(r'\.\. _.*?:', '', texte)
    texte = re.sub(r'<[^>]+>', '', texte)
    return re.sub(r'\s+', ' ', texte).strip()

documents = charger_corpus()
for doc in documents: doc['texte_propre'] = nettoyer_texte_rst(doc['texte'])

def chunk_par_phrases(texte, phrases_par_chunk=3):
    phrases = re.split(r'(?<=[.!?])\s+', texte)
    return [" ".join(phrases[i:i+phrases_par_chunk]) for i in range(0, len(phrases), phrases_par_chunk) if len(" ".join(phrases[i:i+phrases_par_chunk])) > 20]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""])
def chunk_recursif(texte): return splitter.split_text(texte)

model_embed = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def chunk_semantique(texte, seuil_rupture=0.75):
    phrases = [p for p in re.split(r'(?<=[.!?])\s+', texte) if len(p) > 10]
    if len(phrases) < 2: return phrases
    emb = model_embed.encode(phrases)
    chunks, courant = [], [phrases[0]]
    for i in range(1, len(phrases)):
        if cosine_similarity([emb[i-1]], [emb[i]])[0][0] < seuil_rupture:
            chunks.append(" ".join(courant)); courant = [phrases[i]]
        else: courant.append(phrases[i])
    chunks.append(" ".join(courant)); return chunks

strategies = {"Phrases": chunk_par_phrases, "Récursif": chunk_recursif, "Sémantique": chunk_semantique}
resultats_chunking = {}
index_faiss = {}

for nom, fn in strategies.items():
    textes, meta = [], []
    for doc in documents:
        chunks = fn(doc["texte_propre"])
        for i, c in enumerate(chunks):
            textes.append(c); meta.append({"source": doc["source"], "chunk_id": i})
    resultats_chunking[nom] = {"textes": textes, "meta": meta}

    # Index FAISS
    emb = model_embed.encode(textes, normalize_embeddings=True)
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb.astype("float32"))
    index_faiss[nom] = idx
    print(f"✅ Index '{nom}' prêt ({idx.ntotal} chunks)")

def rechercher(question, strategie="Récursif", k=4):
    q_emb = model_embed.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index_faiss[strategie].search(q_emb, k)
    return [{"texte": resultats_chunking[strategie]["textes"][idx], "source": resultats_chunking[strategie]["meta"][idx]["source"], "score": float(score)} for score, idx in zip(scores[0], ids[0])]

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Index 'Phrases' prêt (657 chunks)
✅ Index 'Récursif' prêt (989 chunks)
✅ Index 'Sémantique' prêt (1825 chunks)


In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from tqdm.auto import tqdm
import json

# Modèles
tok_A = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model_A = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.float16, device_map="auto")

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tok_B = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
model_B = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2", quantization_config=bnb, device_map="auto")

nli = pipeline("text-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=0 if torch.cuda.is_available() else -1)

# Fonctions
def generer_reponse(question, passages, modele="A"):
    bloc_passages = "\n\n".join([f"[P{i+1}] {p['texte']}" for i, p in enumerate(passages)])
    system_prompt = "Tu es un assistant documentaire. RÈGLES ABSOLUES : 1. Réponds UNIQUEMENT en FRANÇAIS. 2. Base-toi UNIQUEMENT sur les passages fournis. 3. Si l'info n'y est pas, dis 'Je ne trouve pas cette information dans les documents fournis.' 4. Cite avec [P1], [P2]."
    user_prompt = f"Voici les passages :\n{bloc_passages}\n\nQuestion : {question}\n\nRéponse :"

    tok, mod = (tok_A, model_A) if modele == "A" else (tok_B, model_B)
    prompt_formate = tok.apply_chat_template([{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}], tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt_formate, return_tensors="pt").to(mod.device)

    t0 = time.time()
    out = mod.generate(**inputs, max_new_tokens=250, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip(), time.time() - t0

def detecter_hallucinations(reponse, passages):
    reponse_nettoyee = re.sub(r"```.*?```", " [Code fourni] ", reponse, flags=re.DOTALL)
    reponse_nettoyee = re.sub(r"\[P\d+\]", "", reponse_nettoyee)
    phrases = [p.strip() for p in re.split(r'(?<=[.!?])\s+', reponse_nettoyee) if len(p.strip()) > 15]

    if not phrases: return {"score_fidelite": 1.0, "details": []}

    res = []
    for ph in phrases:
        best_entail = 0.0
        best_passage = None  # <-- LIGNE AJOUTÉE
        for i, p in enumerate(passages):
            pred = nli(f"{p['texte']} </s></s> {ph}")[0]
            if pred["label"].lower() == "entailment" and pred["score"] > best_entail:
                best_entail = pred["score"]
                best_passage = i + 1 # <-- LIGNE AJOUTÉE
        res.append({
            "phrase": ph,
            "soutenue": best_entail >= 0.65,
            "score": round(best_entail, 3),
            "passage_support": best_passage # <-- LIGNE AJOUTÉE
        })
    return {"score_fidelite": round(sum(r["soutenue"] for r in res) / len(res), 3), "details": res}

def repondre_avec_filtre(question, strategie="Récursif", modele="A", k=4):
    passages = rechercher(question, strategie, k=k)
    reponse, latence = generer_reponse(question, passages, modele)
    verif = detecter_hallucinations(reponse, passages)

    decision = "reponse_acceptee"
    if "je ne trouve pas" in reponse.lower() or "i cannot find" in reponse.lower():
        decision = "refus_modele"
    elif verif['score_fidelite'] < 0.60:
        decision = "refus_filtre_nli"
        reponse = "Je ne trouve pas cette information dans les documents fournis (Filtre NLI : risque d'hallucination)."

    return {"question": question, "reponse": reponse, "score_fidelite": verif['score_fidelite'], "latence": latence, "decision": decision}

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [19]:
def evaluer_configuration(df_q, strategie, modele):
    resultats = []
    for _, row in tqdm(df_q.iterrows(), total=len(df_q), desc=f"{strategie}-{modele}"):
        res = repondre_avec_filtre(row['question'], strategie, modele)
        res.update({"id": row['id'], "categorie": row['categorie'], "refus_attendu": row['categorie'] in ("absent", "ambigu")})
        resultats.append(res)
    return resultats

print("Lancement de l'évaluation (cela prend 5-10 min)...")
resultats_A = evaluer_configuration(df_questions, "Récursif", "A")
print("✅ Évaluation terminée !")

Lancement de l'évaluation (cela prend 5-10 min)...


Récursif-A:   0%|          | 0/60 [00:00<?, ?it/s]

✅ Évaluation terminée !


In [6]:
import pandas as pd
import time
from tqdm.auto import tqdm

# 1. Recharger le CSV en empêchant Pandas de transformer les cases vides en nombres (float)
df_questions = pd.read_csv('questions.csv', keep_default_na=False)
print(f"✅ {len(df_questions)} questions rechargées correctement (sans NaN).")

# 2. Redéfinir la fonction de recherche avec la sécurité sur le type "str"
def rechercher(question, strategie="Récursif", k=4):
    question = str(question) # LA CORRECTION : Force la question à être du texte
    q_emb = model_embed.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index_faiss[strategie].search(q_emb, k)

    return [{"texte": resultats_chunking[strategie]["textes"][idx],
             "source": resultats_chunking[strategie]["meta"][idx]["source"],
             "score": float(score)} for score, idx in zip(scores[0], ids[0])]

# 3. Redéfinir la fonction principale avec le filtre
def repondre_avec_filtre(question, strategie="Récursif", modele="A", k=4):
    passages = rechercher(question, strategie, k=k)
    reponse, latence = generer_reponse(question, passages, modele)
    verif = detecter_hallucinations(reponse, passages)

    decision = "reponse_acceptee"
    if "je ne trouve pas" in reponse.lower() or "i cannot find" in reponse.lower():
        decision = "refus_modele"
    elif verif['score_fidelite'] < 0.60:
        decision = "refus_filtre_nli"
        reponse = "Je ne trouve pas cette information dans les documents fournis (Filtre NLI : risque d'hallucination)."

    return {"question": question, "reponse": reponse, "score_fidelite": verif['score_fidelite'], "latence": latence, "decision": decision}

print("✅ Fonctions corrigées et prêtes.")

✅ 60 questions rechargées correctement (sans NaN).
✅ Fonctions corrigées et prêtes.


In [7]:
# 1. La boucle d'évaluation
def evaluer_configuration(df_q, strategie, modele):
    resultats = []
    for _, row in tqdm(df_q.iterrows(), total=len(df_q), desc=f"Évaluation {strategie}-{modele}"):
        res = repondre_avec_filtre(row['question'], strategie, modele)
        res.update({"id": row['id'], "categorie": row['categorie'], "refus_attendu": row['categorie'] in ("absent", "ambigu")})
        resultats.append(res)
    return resultats

print("Lancement de l'évaluation (Modèle A - Qwen)...")
resultats_A_recursif = evaluer_configuration(df_questions, "Récursif", "A")
print("✅ Évaluation terminée !\n")

# 2. Le calcul des métriques finales
def calculer_metriques(resultats):
    df = pd.DataFrame(resultats)

    refus_attendus = df[df["refus_attendu"] == True]
    reponses_attendues = df[df["refus_attendu"] == False]

    taux_refus_justifie = len(refus_attendus[refus_attendus['decision'].str.startswith('refus')]) / max(len(refus_attendus), 1) * 100
    taux_refus_injustifie = len(reponses_attendues[reponses_attendues['decision'].str.startswith('refus')]) / max(len(reponses_attendues), 1) * 100

    return {
        "Fidélité moyenne": round(df["score_fidelite"].mean(), 3),
        "Latence moyenne": f"{df['latence'].mean():.2f}s",
        "Refus justifiés (Attendus)": f"{taux_refus_justifie:.1f} %",
        "Refus injustifiés (Erreurs)": f"{taux_refus_injustifie:.1f} %"
    }

# 3. Affichage
print("--- RÉSULTATS DE L'ÉVALUATION ---")
print("Modèle : Qwen 1.5B | Stratégie : Récursif\n")
metriques = calculer_metriques(resultats_A_recursif)
for k, v in metriques.items():
    print(f"{k:30s} : {v}")

Lancement de l'évaluation (Modèle A - Qwen)...


Évaluation Récursif-A:   0%|          | 0/60 [00:00<?, ?it/s]

✅ Évaluation terminée !

--- RÉSULTATS DE L'ÉVALUATION ---
Modèle : Qwen 1.5B | Stratégie : Récursif

Fidélité moyenne               : 0.015
Latence moyenne                : 1.09s
Refus justifiés (Attendus)     : 0.0 %
Refus injustifiés (Erreurs)    : 100.0 %


In [11]:
import pandas as pd

# On utilise le caractère | comme séparateur, il n'est jamais utilisé dans les questions/réponses
csv_text = """id|question|categorie|reponse_attendue|passages_sources|redacteur
Q01|Comment créer un DataFrame à partir d'un dictionnaire Python ?|present|pd.DataFrame({'A':[1,2],'B':[3,4]})|doc_01|MembreA
Q02|Quelle fonction permet de lire un fichier CSV avec un séparateur personnalisé ?|present|pd.read_csv('fichier.csv', sep=';')|doc_02|MembreB
Q03|Quelle méthode permet de sélectionner les lignes d'un DataFrame par étiquette d'index ?|present|.loc[]|doc_07|MembreC
Q04|Quelle méthode permet de sélectionner des lignes par position numérique ?|present|.iloc[]|doc_07|MembreA
Q05|Quelle fonction permet de fusionner deux DataFrames sur une clé commune ?|present|pd.merge(left, right, on='clé')|doc_08|MembreB
Q06|Comment calculer la moyenne d'une colonne numérique ?|present|df['col'].mean()|doc_05|MembreC
Q07|Comment grouper un DataFrame par une colonne et calculer une agrégation ?|present|df.groupby('col').agg(...)|doc_09|MembreA
Q08|Comment remplacer les valeurs manquantes d'un DataFrame ?|present|df.fillna(valeur) ou df.dropna()|doc_11|MembreB
Q09|Comment pivoter un DataFrame du format long vers le format large ?|present|df.pivot() ou df.pivot_table()|doc_10|MembreC
Q10|Comment créer un graphique à partir d'un DataFrame ?|present|df.plot()|doc_04|MembreA
Q11|Comment exporter un DataFrame vers un fichier Excel ?|present|df.to_excel('fichier.xlsx')|doc_02|MembreB
Q12|Comment trier un DataFrame selon une colonne ?|present|df.sort_values(by='col')|doc_05|MembreC
Q13|Comment concaténer deux DataFrames verticalement ?|present|pd.concat([df1, df2], axis=0)|doc_08|MembreA
Q14|Comment filtrer les lignes dont une colonne dépasse un seuil ?|present|df[df['col'] > seuil]|doc_07|MembreB
Q15|Quelle fonction permet de lire un fichier JSON dans pandas ?|present|pd.read_json('fichier.json')|doc_06|MembreC
Q16|Comment renommer les colonnes d'un DataFrame ?|present|df.rename(columns={'ancien':'nouveau'})|doc_05|MembreA
Q17|Comment convertir une colonne en type datetime ?|present|pd.to_datetime(df['col'])|doc_12|MembreB
Q18|Comment compter le nombre de valeurs uniques dans une colonne ?|present|df['col'].nunique()|doc_05|MembreC
Q19|Quelle option permet d'afficher plus de colonnes dans un DataFrame ?|present|pd.set_option('display.max_columns', N)|doc_14|MembreA
Q20|Comment supprimer une colonne d'un DataFrame ?|present|df.drop(columns=['col'])|doc_05|MembreB
Q21|Quelle est la politique de pandas concernant le support de MongoDB ?|absent|||MembreC
Q22|Quel est le salaire moyen d'un développeur pandas en France ?|absent|||MembreA
Q23|Quelle est la météo à Paris le jour de la sortie de pandas 2.0 ?|absent|||MembreB
Q24|Comment installer pandas sur un ordinateur quantique ?|absent|||MembreC
Q25|Quel est le nombre exact de contributeurs au projet pandas en 2020 ?|absent|||MembreA
Q26|Quelle est la durée légale du travail pour un data scientist ?|absent|||MembreB
Q27|Comment configurer pandas pour utiliser un GPU AMD ?|absent|||MembreC
Q28|Quelle est la capitale de l'Australie ?|absent|||MembreA
Q29|Quelle est la différence entre pandas et le framework Spark RDD ?|absent|||MembreB
Q30|Quel est l'impact carbone de l'exécution d'un script pandas ?|absent|||MembreC
Q31|Comment récupérer les données de la Bourse de Casablanca en temps réel ?|absent|||MembreA
Q32|Quel est le prix d'une licence commerciale pandas ?|absent|||MembreB
Q33|Comment intégrer pandas avec un modèle de deep learning dans TensorFlow ?|absent|||MembreC
Q34|Quelle est l'origine du nom "pandas" ?|absent|||MembreA
Q35|Comment fonctionne la gestion mémoire en C sous-jacente à pandas ?|absent|||MembreB
Q36|Comment faire ?|ambigu|||MembreC
Q37|Pourquoi ça ne marche pas ?|ambigu|||MembreA
Q38|Quelle est la meilleure façon de procéder ?|ambigu|||MembreB
Q39|Comment améliorer les performances ?|ambigu|||MembreC
Q40|Comment corriger cette erreur ?|ambigu|||MembreA
Q41|C'est quoi le problème avec mes données ?|ambigu|||MembreB
Q42|Comment utiliser cette fonction ?|ambigu|||MembreC
Q43|Comment faire pour que ce soit plus rapide ?|ambigu|||MembreA
Q44|Quelle syntaxe utiliser ?|ambigu|||MembreB
Q45|Comment gérer ce cas ?|ambigu|||MembreC
Q46|Comment lire un fichier CSV puis grouper les données par une colonne et calculer la moyenne d'une autre colonne ?|multi|pd.read_csv + groupby().mean()|doc_02 + doc_09|MembreA
Q47|Comparer les méthodes .loc et .iloc pour la sélection de lignes et de colonnes dans un DataFrame|multi|.loc = labels; .iloc = positions|doc_05 + doc_07|MembreB
Q48|Comment gérer les valeurs manquantes dans un DataFrame avant de générer un graphique ?|multi|fillna/dropna puis .plot()|doc_04 + doc_11|MembreC
Q49|Comment fusionner deux DataFrames puis pivoter le résultat pour produire un tableau croisé ?|multi|merge() puis pivot_table()|doc_08 + doc_10|MembreA
Q50|Comment convertir une chaîne en date puis extraire l'année et agréger par mois ?|multi|to_datetime puis .dt.year et resample|doc_09 + doc_12|MembreB
Q51|Comment exporter un DataFrame en CSV puis le recharger en préservant les types numériques et les dates ?|multi|to_csv + read_csv(parse_dates=..., dtype=...)|doc_02 + doc_06|MembreC
Q52|Comment détecter les doublons après une fusion de deux DataFrames et les supprimer ?|multi|merge + duplicated + drop_duplicates|doc_08 + doc_11|MembreA
Q53|Comment visualiser l'évolution temporelle d'une série groupée par catégorie ?|multi|groupby + .plot()|doc_09 + doc_13|MembreB
Q54|Comment concaténer deux DataFrames de structures différentes puis remplacer les NaN résultants ?|multi|pd.concat + fillna|doc_08 + doc_11|MembreC
Q55|Comment pivoter un DataFrame puis tracer un graphique en barres empilées ?|multi|pivot_table puis .plot(kind='bar')|doc_10 + doc_13|MembreA
Q56|Comment rééchantillonner une série temporelle et calculer la moyenne par mois ?|multi|resample('M').mean()|doc_09 + doc_12|MembreB
Q57|Comment filtrer un DataFrame sur plusieurs conditions et vérifier les valeurs manquantes du sous-ensemble ?|multi|df[(cond1) & (cond2)] puis .isna().sum()|doc_07 + doc_11|MembreC
Q58|Comment renommer les colonnes d'un DataFrame puis les utiliser comme index ?|multi|rename() puis set_index()|doc_05 + doc_07|MembreA
Q59|Comment lire un fichier JSON volumineux et le transformer en DataFrame pour une analyse groupée ?|multi|read_json + groupby|doc_06 + doc_09|MembreB
Q60|Comment désactiver les avertissements de pandas et afficher toutes les colonnes pour le débogage ?|multi|warnings.filterwarnings + pd.set_option|doc_05 + doc_14|MembreC"""

with open('questions.csv', 'w', encoding='utf-8') as f:
    f.write(csv_text)

# LECTURE AVEC LE SÉPARATEUR |
df_questions = pd.read_csv('questions.csv', sep='|', keep_default_na=False)
print("✅ Fichier CSV corrigé et rechargé !")
print(f"Nombre de questions : {len(df_questions)}")
# On affiche les 2 premières lignes pour vérifier
print(df_questions.head(2))

✅ Fichier CSV corrigé et rechargé !
Nombre de questions : 60
    id                                           question categorie  \
0  Q01  Comment créer un DataFrame à partir d'un dicti...   present   
1  Q02  Quelle fonction permet de lire un fichier CSV ...   present   

                      reponse_attendue passages_sources redacteur  
0  pd.DataFrame({'A':[1,2],'B':[3,4]})           doc_01   MembreA  
1  pd.read_csv('fichier.csv', sep=';')           doc_02   MembreB  


In [12]:
import time
from tqdm.auto import tqdm

# 1. Correction de la fonction NLI
def detecter_hallucinations(reponse, passages):
    import re
    reponse_nettoyee = re.sub(r"```.*?```", " [Code fourni] ", reponse, flags=re.DOTALL)
    reponse_nettoyee = re.sub(r"\[P\d+\]", "", reponse_nettoyee)
    phrases = [p.strip() for p in re.split(r'(?<=[.!?])\s+', reponse_nettoyee) if len(p.strip()) > 15]

    if not phrases: return {"score_fidelite": 1.0, "details": []}

    res = []
    for ph in phrases:
        best_entail = 0.0
        best_passage = None
        for i, p in enumerate(passages):
            pred = nli(f"{p['texte']} </s></s> {ph}")[0]
            if pred["label"].lower() == "entailment" and pred["score"] > best_entail:
                best_entail = pred["score"]
                best_passage = i + 1
        res.append({
            "phrase": ph,
            "soutenue": best_entail >= 0.60,
            "score": round(best_entail, 3),
            "passage_support": best_passage
        })
    return {"score_fidelite": round(sum(r["soutenue"] for r in res) / len(res), 3), "details": res}

# 2. Correction du filtre
def repondre_avec_filtre(question, strategie="Récursif", modele="A", k=4):
    passages = rechercher(question, strategie, k=k)
    reponse, latence = generer_reponse(question, passages, modele)
    verif = detecter_hallucinations(reponse, passages)

    decision = "reponse_acceptee"
    if "je ne trouve pas" in reponse.lower() or "i cannot find" in reponse.lower():
        decision = "refus_modele"
    elif verif['score_fidelite'] < 0.60:
        decision = "refus_filtre_nli"
        reponse = "Je ne trouve pas cette information dans les documents fournis (Filtre NLI : risque d'hallucination)."

    return {"question": question, "reponse": reponse, "score_fidelite": verif['score_fidelite'], "latence": latence, "decision": decision}

# 3. Lancement de l'évaluation
def evaluer_configuration(df_q, strategie, modele):
    resultats = []
    for _, row in tqdm(df_q.iterrows(), total=len(df_q), desc=f"Évaluation {strategie}-{modele}"):
        res = repondre_avec_filtre(row['question'], strategie, modele)
        res.update({"id": row['id'], "categorie": row['categorie'], "refus_attendu": row['categorie'] in ("absent", "ambigu")})
        resultats.append(res)
    return resultats

print("Lancement de l'évaluation complète (Modèle Qwen)...")
resultats_A = evaluer_configuration(df_questions, "Récursif", "A")
print("✅ Terminé !\n")

# 4. Affichage des métriques
def calculer_metriques(resultats):
    df = pd.DataFrame(resultats)
    refus_attendus = df[df["refus_attendu"] == True]
    reponses_attendues = df[df["refus_attendu"] == False]

    taux_justifie = len(refus_attendus[refus_attendus['decision'].str.startswith('refus')]) / max(len(refus_attendus), 1) * 100
    taux_injustifie = len(reponses_attendues[reponses_attendues['decision'].str.startswith('refus')]) / max(len(reponses_attendues), 1) * 100

    return {
        "Fidélité moyenne": round(df["score_fidelite"].mean(), 3),
        "Latence moyenne": f"{df['latence'].mean():.2f}s",
        "Refus justifiés (Attendus)": f"{taux_justifie:.1f} %",
        "Refus injustifiés (Erreurs)": f"{taux_injustifie:.1f} %"
    }

print("--- RÉSULTATS FINAUX (Qwen 1.5B | Récursif) ---")
for k, v in calculer_metriques(resultats_A).items():
    print(f"{k:30s} : {v}")

Lancement de l'évaluation complète (Modèle Qwen)...


Évaluation Récursif-A:   0%|          | 0/60 [00:00<?, ?it/s]

✅ Terminé !

--- RÉSULTATS FINAUX (Qwen 1.5B | Récursif) ---
Fidélité moyenne               : 0.409
Latence moyenne                : 4.81s
Refus justifiés (Attendus)     : 88.0 %
Refus injustifiés (Erreurs)    : 54.3 %


In [13]:
import numpy as np

print("--- ÉTUDE DE SENSIBILITÉ DU SEUIL NLI ---")
print(f"{'Seuil':<10} | {'Refus Justifiés':<20} | {'Refus Injustifiés':<20}")
print("-" * 55)

# On teste des seuils de 0.30 à 0.80
for seuil in np.arange(0.30, 0.85, 0.10):
    refus_justifies, refus_injustifies = 0, 0
    total_attendus, total_reponses = 0, 0

    for res in resultats_A:
        # On simule la décision du système avec le nouveau seuil
        est_refus = res['score_fidelite'] < seuil or "je ne trouve pas" in res['reponse'].lower()

        if res['refus_attendu']:
            total_attendus += 1
            if est_refus: refus_justifies += 1
        else:
            total_reponses += 1
            if est_refus: refus_injustifies += 1

    tx_just = (refus_justifies / total_attendus) * 100
    tx_injust = (refus_injustifies / total_reponses) * 100

    print(f"{seuil:<10.2f} | {tx_just:>15.1f} % | {tx_injust:>17.1f} %")

--- ÉTUDE DE SENSIBILITÉ DU SEUIL NLI ---
Seuil      | Refus Justifiés      | Refus Injustifiés   
-------------------------------------------------------
0.30       |            88.0 % |              54.3 %
0.40       |            88.0 % |              54.3 %
0.50       |            88.0 % |              54.3 %
0.60       |            92.0 % |              57.1 %
0.70       |            92.0 % |              60.0 %
0.80       |            92.0 % |              68.6 %


In [14]:
print("Lancement de l'évaluation complète (Modèle B - Mistral)...")
# On utilise la même fonction d'évaluation, mais avec modele="B"
resultats_B = evaluer_configuration(df_questions, "Récursif", "B")
print("✅ Terminé !\n")

print("--- RÉSULTATS FINAUX (Mistral 7B | Récursif) ---")
for k, v in calculer_metriques(resultats_B).items():
    print(f"{k:30s} : {v}")

Lancement de l'évaluation complète (Modèle B - Mistral)...


Évaluation Récursif-B:   0%|          | 0/60 [00:00<?, ?it/s]

✅ Terminé !

--- RÉSULTATS FINAUX (Mistral 7B | Récursif) ---
Fidélité moyenne               : 0.319
Latence moyenne                : 14.61s
Refus justifiés (Attendus)     : 92.0 %
Refus injustifiés (Erreurs)    : 68.6 %


In [15]:
import pandas as pd

print("--- ANALYSE D'ERREURS (Modèle Mistral) ---\n")

# On cherche des exemples de refus injustifiés (le bot a refusé alors qu'il ne fallait pas)
erreurs_refus = [r for r in resultats_B if r['refus_attendu'] == False and r['decision'].startswith('refus')]

for i, err in enumerate(erreurs_refus[:3]):
    print(f"❌ ERREUR {i+1} (ID: {err['id']} | Catégorie: {err['categorie']})")
    print(f"Question : {err['question']}")
    print(f"Réponse du bot : {err['reponse'][:200]}...")
    print(f"Score de fidélité : {err['score_fidelite']}")
    print("-" * 50)

# On cherche des exemples de hallucinations non filtrées
print("\n--- HALLUCINATIONS NON FILTRÉES ---\n")
erreurs_hallu = [r for r in resultats_B if r['refus_attendu'] == True and not r['decision'].startswith('refus')]

if not erreurs_hallu:
    print("Aucune hallucination n'est passée à travers le filtre ! (Le filtre a été trop strict).")
else:
    for i, err in enumerate(erreurs_hallu[:3]):
        print(f"❌ HALLUCINATION {i+1} (ID: {err['id']} | Catégorie: {err['categorie']})")
        print(f"Question : {err['question']}")
        print(f"Réponse du bot : {err['reponse'][:200]}...")
        print(f"Score de fidélité : {err['score_fidelite']}")
        print("-" * 50)

--- ANALYSE D'ERREURS (Modèle Mistral) ---

❌ ERREUR 1 (ID: Q01 | Catégorie: present)
Question : Comment créer un DataFrame à partir d'un dictionnaire Python ?
Réponse du bot : Je ne trouve pas cette information dans les documents fournis (Filtre NLI : risque d'hallucination)....
Score de fidélité : 0.0
--------------------------------------------------
❌ ERREUR 2 (ID: Q06 | Catégorie: present)
Question : Comment calculer la moyenne d'une colonne numérique ?
Réponse du bot : Je ne trouve pas cette information dans les documents fournis (Filtre NLI : risque d'hallucination)....
Score de fidélité : 0.5
--------------------------------------------------
❌ ERREUR 3 (ID: Q07 | Catégorie: present)
Question : Comment grouper un DataFrame par une colonne et calculer une agrégation ?
Réponse du bot : Je ne trouve pas cette information dans les documents fournis (Filtre NLI : risque d'hallucination)....
Score de fidélité : 0.0
--------------------------------------------------

--- HALLUCINATION

In [16]:
print("Test des autres stratégies de chunking avec Qwen (rapide)...\n")

# 1. Test avec Phrases
print("1/2 : Évaluation avec la stratégie 'Phrases'...")
resultats_A_phrases = evaluer_configuration(df_questions, "Phrases", "A")
metriques_phrases = calculer_metriques(resultats_A_phrases)

# 2. Test avec Sémantique
print("\n2/2 : Évaluation avec la stratégie 'Sémantique'...")
resultats_A_semantique = evaluer_configuration(df_questions, "Sémantique", "A")
metriques_semantique = calculer_metriques(resultats_A_semantique)

# 3. Tableau comparatif final
print("\n" + "="*60)
print("TABLEAU COMPARATIF DES STRATÉGIES DE CHUNKING (Modèle Qwen)")
print("="*60)

tableau = pd.DataFrame({
    "Récursif (Base)": calculer_metriques(resultats_A),
    "Phrases": metriques_phrases,
    "Sémantique": metriques_semantique
}).T

display(tableau)

Test des autres stratégies de chunking avec Qwen (rapide)...

1/2 : Évaluation avec la stratégie 'Phrases'...


Évaluation Phrases-A:   0%|          | 0/60 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (548 > 512). Running this sequence through the model will result in indexing errors



2/2 : Évaluation avec la stratégie 'Sémantique'...


Évaluation Sémantique-A:   0%|          | 0/60 [00:00<?, ?it/s]


TABLEAU COMPARATIF DES STRATÉGIES DE CHUNKING (Modèle Qwen)


,Fidélité moyenne,Latence moyenne,Refus justifiés (Attendus),Refus injustifiés (Erreurs)
Récursif (Base),0.409,4.81s,88.0 %,54.3 %
Phrases,0.316,5.15s,92.0 %,68.6 %
Sémantique,0.352,4.41s,84.0 %,57.1 %
